# Foundational Models and Agents

This notebook is a practical reference for the first building blocks of a LangChain application. It moves from a direct chat-model call to configurable providers, then shows how an agent packages messages and how streaming exposes incremental output.

## Learning goals

- Load environment variables and initialize a chat model.
- Inspect model responses and response metadata.
- Change model configuration and provider implementations.
- Create an agent, provide conversation history, and stream its output.

The examples use a deliberately simple question so the mechanics remain visible. API keys are loaded from the repository `.env` file; model names and provider availability may change over time.

In [ ]:
# Load API keys and other settings from the repository's .env file.
from dotenv import load_dotenv
load_dotenv()

True

## Initializing and Invoking a model

### What to notice

A chat model is a runnable interface: pass it a prompt with `invoke`, then inspect the returned `AIMessage`. The message contains user-facing content plus metadata that can help with tracing, token accounting, and provider-specific diagnostics.

In [ ]:
from langchain.chat_models import init_chat_model

# init_chat_model returns a provider-aware chat model runnable.
model = init_chat_model(model = "gpt-5-nano")

In [3]:
response = model.invoke("What's the capital of the Moon?")
response

AIMessage(content='There isn’t one. The Moon isn’t a country or a government, so it has no capital. Any human activity there would be under the authority of the organizing country or organization, not a moon-wide capital. If you’re writing fiction, you could invent a capital name (e.g., Luna City, Selene) for your story.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 590, 'prompt_tokens': 13, 'total_tokens': 603, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 512, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EPU7LM1zWXiZc0EkzhIoZqDtFkf0f', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0b4f9-fa81-7432-92ef-8dd7ec616c2a-0', tool_calls=[], invalid_tool_call

In [4]:
print(response.content)

There isn’t one. The Moon isn’t a country or a government, so it has no capital. Any human activity there would be under the authority of the organizing country or organization, not a moon-wide capital. If you’re writing fiction, you could invent a capital name (e.g., Luna City, Selene) for your story.


In [5]:
from pprint import pprint

pprint(response.response_metadata)

{'finish_reason': 'stop',
 'id': 'chatcmpl-EPU7LM1zWXiZc0EkzhIoZqDtFkf0f',
 'logprobs': None,
 'model_name': 'gpt-5-nano-2025-08-07',
 'model_provider': 'openai',
 'service_tier': 'default',
 'system_fingerprint': None,
 'token_usage': {'completion_tokens': 590,
                 'completion_tokens_details': {'accepted_prediction_tokens': 0,
                                               'audio_tokens': 0,
                                               'reasoning_tokens': 512,
                                               'rejected_prediction_tokens': 0},
                 'prompt_tokens': 13,
                 'prompt_tokens_details': {'audio_tokens': 0,
                                           'cache_write_tokens': None,
                                           'cached_tokens': 0},
                 'total_tokens': 603}}


## Customizing your model

### What to notice

Model configuration belongs at initialization time. Parameters such as `temperature` affect how responses are generated, so keep the model setup in one visible place when comparing experiments.

In [ ]:
model = init_chat_model(
    model = "gpt-5-nano",
    # Higher temperature generally allows more variation in generated text.
    temperature = 1.0)

response = model.invoke("What's the capital of the Moon?")
print(response.content)

There isn’t one. The Moon isn’t a country or government, so it has no capital. If you’re imagining a fictional lunar colony, you might call its capital something like “Lunapolis,” “Selene City,” or any name you choose. If you’re writing a story or worldbuilding, I can help brainstorm ideas.


## Model Providers
https://docs.langchain.com/oss/python/integrations/chat

### Provider comparison

`init_chat_model` gives the notebook a common interface while the provider-specific class demonstrates an explicit integration. In production, confirm the provider package is installed and the matching API key is available before switching models.

In [7]:
model = init_chat_model(model = "claude-sonnet-4-6")

response = model.invoke("What's the capital of the Moon?")
print(response.content)

The Moon doesn't have a capital city. It has no permanent human settlements, governments, or administrative divisions. Only 12 astronauts have ever walked on its surface, all during NASA's Apollo missions (1969–1972), and none stayed permanently.


In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")
response = model.invoke("What's the capital of the Moon?")
print(response.content)

[{'type': 'text', 'text': 'The Moon does not have a capital because it is not a country and has no permanent residents or government. \n\nAccording to the **Outer Space Treaty of 1967**, which has been signed by all major spacefaring nations, no country can claim sovereignty over the Moon or any other celestial body. Therefore, there are no cities, borders, or capital cities on the lunar surface.', 'extras': {'signature': 'EnEKbwFpFH0T5ZybMbj2CXmLFGaSoBL5evFJNZdgDTus7zLxcdeMC4942BgH1Bj6HaU7pCj9uaIkfQ50s6aVHaDXl2WQfWhWByJZQcm7+Jql9OfMISSaCCNy4BnH/L5KMDfktZLjfA6qIqpj7N58z5WeOw=='}}]


## Initializing and invoking an agent

### From a model call to an agent

An agent adds a message-based state contract around the model. The examples below show the simplest input shape, how to inspect the returned message list, and how prior `HumanMessage` and `AIMessage` objects provide conversation context.

In [ ]:
from langchain.agents import create_agent

# An agent uses the model and exposes conversation state through messages.
agent = create_agent(model = model)

In [14]:
agent = create_agent(model = "claude-sonnet-4-6")

In [15]:
agent = create_agent("gpt-5-nano")

In [ ]:
from langchain.messages import HumanMessage

# Agents receive state as a dictionary whose messages list is the conversation.
response = agent.invoke({
    "messages": [HumanMessage(content="What's the capital of the Moon?")]
})
print(response)

{'messages': [HumanMessage(content="What's the capital of the Moon?", additional_kwargs={}, response_metadata={}, id='3a5eb462-f0df-490a-bd9a-575b4379efda'), AIMessage(content='There isn’t one. The Moon has no government or permanent population, so it has no capital. Under the Outer Space Treaty, no country can claim sovereignty over the Moon, and any future settlements would require international agreements. If you’re thinking fiction, you could pick a fictional capital like “Lunapolis,” “Selene City,” or “Artemis Prime.”', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 659, 'prompt_tokens': 13, 'total_tokens': 672, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 576, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None

In [17]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content="What's the capital of the Moon?", additional_kwargs={}, response_metadata={}, id='5ae29100-4047-4748-a2e8-4db7befce1f8'),
              AIMessage(content='There isn’t one. The Moon isn’t a country or sovereign entity, so it has no capital. If you’re asking about fiction or a hypothetical future, tell me the work or scenario and I can share what they call a capital.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 377, 'prompt_tokens': 13, 'total_tokens': 390, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EPUeUcZ5ndwstKkhRQ6aU6qYcAkn9', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='

In [19]:
print(response['messages'][-1].content)

There isn’t one. The Moon has no government or permanent population, so it has no capital. Under the Outer Space Treaty, no country can claim sovereignty over the Moon, and any future settlements would require international agreements. If you’re thinking fiction, you could pick a fictional capital like “Lunapolis,” “Selene City,” or “Artemis Prime.”


In [20]:
from langchain.messages import AIMessage

response = agent.invoke({
    "messages": [HumanMessage(content = "What's the capital of the Moon?"),
    AIMessage(content = "The capital of the Moon is Luna City."),
    HumanMessage(content = "Interesting, tell me more about Luna City.")
    ]})
pprint(response)

{'messages': [HumanMessage(content="What's the capital of the Moon?", additional_kwargs={}, response_metadata={}, id='9cd138f1-5e10-40d5-86d3-9001c45a8f02'),
              AIMessage(content='The capital of the Moon is Luna City.', additional_kwargs={}, response_metadata={}, id='42c9c2e3-6e49-4155-ab39-fa4238444308', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content='Interesting, tell me more about Luna City.', additional_kwargs={}, response_metadata={}, id='59324f5f-5278-4392-a380-e84968765fb4'),
              AIMessage(content='Nice to hear you’re curious about a fictional Luna City. Since there’s no real capital of the Moon, here’s a rich, self-contained idea you can use for worldbuilding, stories, games, or just for fun.\n\nOverview\n- Where it sits: Luna City is built in a sunlit crater near the Moon’s southern limb, shielded from the harsh extremes by tall crater walls. The city is a network of inflatable and rigid-shell domes connected by pressurized tunne

## Streaming Output

### Streaming summary

`stream(..., stream_mode="messages")` yields message chunks as they arrive. The metadata tuple identifies the producing node, while `token.content` is the incremental text to render. This pattern is useful for responsive interfaces and long-running agent work.

In [ ]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="Tell me all about Luna City, the capital of the Moon")]},
    stream_mode="messages"
):

    # Each iteration provides an incremental message chunk and its source metadata.
    if token.content:
        print(token.content, end="", flush=True)

There isn’t a real Luna City today—the Moon has no capital. If you’re imagining a fictional setting, here’s a full, ready-to-use concept for Luna City as the Moon’s capital. It includes geography, government, economy, culture, tech, and landmarks you can drop into a story, game, or world-building project.

Big picture
- Name: Luna City (officially the Capital of the Lunar Confederation)
- Role: Political heart, cultural as well as scientific hub for lunar settlements; home to the main government complex, major institutions, and the orbital-comm infrastructure that links Earth and the Moon.
- Tone: Hard sci-fi with grounded tech, but room for epic moments and character-driven stories. Think bureaucratic diplomacy next to dazzling solar skylines and ice vaults.

Snapshot
- Population: 2–3 million residents (a dense, self-contained ecology of domes, tunnels, and rings)
- Location: A permanently sunlit rim around a polar crater (built around Shackleton-like terrain or a similar pole-crater

## Conclusion and reuse checklist

The core progression is:

1. Load credentials once at the start of the notebook.
2. Initialize a chat model through a stable interface.
3. Use `invoke` for a complete response and inspect both content and metadata.
4. Tune model parameters deliberately and document the choice.
5. Swap providers without changing the surrounding message workflow.
6. Use an agent when the application needs message state or tools.
7. Use streaming when the caller benefits from partial results.

For future experiments, change one variable at a time: the model, provider, prompt, configuration, or output mode. That makes response differences easier to explain and gives this notebook a dependable baseline.